# 03 - Personal recovery trend analysis

This notebook reviews the transparent recovery score produced by `src/recovery_score.py`. The score is for personal trend exploration only. It is not a medical diagnosis, disease prediction, or substitute for professional care.

Fixed weights: sleep 30%, HRV 30%, resting heart rate 20%, previous-day exercise minutes 10%, and previous-day active energy 10%.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.visualize import plot_recovery_trend, save_figure_formats

SCORE_PATH = PROJECT_ROOT / "data" / "processed" / "daily_recovery_score.csv"

scores = pd.read_csv(SCORE_PATH, parse_dates=["date"])
print(f"Rows: {len(scores)}")
print(f"Scored days: {scores['recovery_score'].notna().sum()}")

## Score coverage

`weight_coverage` reports how much of the fixed weighting was available each day. The final score is left missing when coverage is below 40% or all core recovery signals (sleep, HRV, and resting heart rate) are missing.

In [ ]:
coverage_summary = pd.DataFrame({
    "missing_count": scores.isna().sum(),
    "missing_rate": scores.isna().mean(),
})
coverage_summary

## Recovery trend

The rolling line smooths short-term variation. It should be interpreted as a retrospective personal trend, not a clinical threshold.

In [ ]:
df = scores.copy()
df["date"] = pd.to_datetime(df["date"])

start_date = "2025-06-01"
end_date = "2026-01-01"
df = df[(df["date"] >= start_date) & (df["date"] < end_date)]

fig = plot_recovery_trend(df, version="portfolio")
ax = fig.axes[0]
ax.set_xlim(
    pd.Timestamp(start_date),
    pd.Timestamp(end_date) - pd.Timedelta(days=1),
)
ax.set_title(
    "Personal Recovery Score Trend: Jun–Dec 2025",
    loc="left",
    fontsize=20,
    fontweight="bold",
    color="#22313F",
    pad=42,
)

for text in list(ax.texts):
    if "Apple Watch / Apple Health daily metrics" in text.get_text():
        text.remove()

scored_days = int(df["recovery_score"].notna().sum())
ax.text(
    0,
    1.035,
    (
        "2025-06-01 to 2025-12-31  |  "
        f"{scored_days:,} scored days  |  "
        "14-day rolling window  |  Apple Watch / Apple Health daily metrics"
    ),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10,
    color="#657786",
)

figure_path = (
    PROJECT_ROOT
    / "reports"
    / "figures"
    / "recovery_score_trend_2025_06_12.png"
)
save_figure_formats(fig, figure_path, dpi=240)
plt.show()

## Component relationships

Use this view to understand which available components move with the total score. Correlation does not establish causation.

In [ ]:
component_columns = [
    "recovery_score",
    "sleep_component_score",
    "hrv_component_score",
    "resting_hr_component_score",
    "previous_exercise_load_component_score",
    "previous_active_energy_load_component_score",
]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(scores[component_columns].corr(), cmap="vlag", center=0, ax=ax)
ax.set_title("Recovery Score Component Correlations")
plt.show()